# Task 2.2 — Implementation of Core Contribution


## Setup and Imports

In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## Step 1: Load Data and Build Pairs

We reuse the pair construction from Task 2.1. This generates the binary same/different-person dataset described in **Section 2, Equation (1)** of the paper.

In [2]:
data = fetch_olivetti_faces(shuffle=False)
X_raw = data.data
y_id  = data.target

def build_pairs(X, y, n_pos=1000, n_neg=1000, random_state=42):
    """Construct same/different-identity pairs. Section 2, Eq. (1)."""
    rng = np.random.RandomState(random_state)
    pa, pb, labels = [], [], []
    ids = np.unique(y)
    for _ in range(n_pos):
        idn = rng.choice(ids)
        imgs = np.where(y == idn)[0]
        i, j = rng.choice(imgs, size=2, replace=False)
        pa.append(X[i]); pb.append(X[j]); labels.append(1)
    for _ in range(n_neg):
        id1, id2 = rng.choice(ids, size=2, replace=False)
        i = rng.choice(np.where(y == id1)[0])
        j = rng.choice(np.where(y == id2)[0])
        pa.append(X[i]); pb.append(X[j]); labels.append(-1)
    return np.array(pa), np.array(pb), np.array(labels)

# Reduce dimensionality via PCA for tractable SVM training on a laptop
# The paper uses raw SIFT/LBP features; we use PCA-compressed pixels as a proxy
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_raw)  # (400, 50)
print(f"PCA: {X_raw.shape[1]} dims → {X_pca.shape[1]} dims "
      f"(explained variance: {pca.explained_variance_ratio_.sum():.2%})")

pairs_a, pairs_b, pair_labels = build_pairs(X_pca, y_id, n_pos=1000, n_neg=1000)
print(f"Pairs: {len(pair_labels)} total, "
      f"+1: {(pair_labels==1).sum()}, -1: {(pair_labels==-1).sum()}")

PCA: 4096 dims → 50 dims (explained variance: 87.34%)


Pairs: 2000 total, +1: 1000, -1: 1000


**Why PCA?** The original 4096-d features make the kernel matrix computation $O(n^2 \times d)$ which is slow on a laptop. PCA with 50 components retains most variance while keeping training time in seconds. This is a toy-dataset compromise absent from the original paper.

## Step 2: Implement the Balanced Pairwise Kernel (K_DL)

This is the **core contribution** of the paper — the Direct Linearised (balanced) kernel defined in **Section 3, Equation (6)**:

$$K_{DL}((a,b),(c,d)) = k(a,c) + k(b,d) + k(a,d) + k(b,c)$$

where $k$ is a standard base kernel (we use RBF). The cross-terms $k(a,d)$ and $k(b,c)$ are the key addition over $K_D$ that enforce symmetry (Theorem 2).

In [3]:
from sklearn.metrics.pairwise import rbf_kernel

def compute_K_DL(A1, B1, A2, B2, gamma=0.01):
    """
    Compute the Direct Linearised (Balanced) Pairwise Kernel matrix.
    
    K_DL((a,b),(c,d)) = k(a,c) + k(b,d) + k(a,d) + k(b,c)
    Reference: Brunner et al. (2012), Section 3, Equation (6)
    
    Parameters:
        A1, B1: first-side and second-side features for set 1 (rows)
        A2, B2: first-side and second-side features for set 2 (cols)
        gamma: RBF kernel bandwidth parameter
    Returns:
        K: kernel matrix of shape (len(A1), len(A2))
    """
    kac = rbf_kernel(A1, A2, gamma=gamma)  # k(a, c)
    kbd = rbf_kernel(B1, B2, gamma=gamma)  # k(b, d)
    kad = rbf_kernel(A1, B2, gamma=gamma)  # k(a, d)  ← cross term 1
    kbc = rbf_kernel(B1, A2, gamma=gamma)  # k(b, c)  ← cross term 2
    return kac + kbd + kad + kbc           # Eq. (6)


def compute_K_D(A1, B1, A2, B2, gamma=0.01):
    """
    Compute the plain Direct Pairwise Kernel (baseline, NOT symmetric).
    K_D((a,b),(c,d)) = k(a,c) + k(b,d)
    Reference: Brunner et al. (2012), Section 2, Equation (4)
    """
    kac = rbf_kernel(A1, A2, gamma=gamma)
    kbd = rbf_kernel(B1, B2, gamma=gamma)
    return kac + kbd                       # Eq. (4)


print("Pairwise kernel functions defined.")
print("K_DL reference: Brunner et al. (2012), Section 3, Equation (6)")
print("K_D reference:  Brunner et al. (2012), Section 2, Equation (4)")

Pairwise kernel functions defined.
K_DL reference: Brunner et al. (2012), Section 3, Equation (6)
K_D reference:  Brunner et al. (2012), Section 2, Equation (4)


**`compute_K_DL`** implements Equation (6) of the paper. The four calls to `rbf_kernel` correspond to the four terms in $K_{DL}$. The cross-terms `kad` and `kbc` are the difference from the plain $K_D$ — they mix the first element of one pair with the second element of another, which is what enforces the symmetry property stated in **Theorem 2**.

## Step 3: Train-Test Split and Kernel Matrix Computation

In [4]:
# Train/test split
idx = np.arange(len(pair_labels))
idx_tr, idx_te = train_test_split(idx, test_size=0.25, random_state=42,
                                   stratify=pair_labels)

A_tr, B_tr, y_tr = pairs_a[idx_tr], pairs_b[idx_tr], pair_labels[idx_tr]
A_te, B_te, y_te = pairs_a[idx_te], pairs_b[idx_te], pair_labels[idx_te]

GAMMA = 0.01  # RBF bandwidth; tuned manually

# Compute kernel matrices — Section 4 describes caching k evaluations
print("Computing K_DL kernel matrices...")
K_DL_train = compute_K_DL(A_tr, B_tr, A_tr, B_tr, gamma=GAMMA)
K_DL_test  = compute_K_DL(A_te, B_te, A_tr, B_tr, gamma=GAMMA)

print("Computing K_D kernel matrices...")
K_D_train  = compute_K_D(A_tr, B_tr, A_tr, B_tr, gamma=GAMMA)
K_D_test   = compute_K_D(A_te, B_te, A_tr, B_tr, gamma=GAMMA)

print(f"Train kernel matrix shape: {K_DL_train.shape}")
print(f"Test kernel matrix shape:  {K_DL_test.shape}")

Computing K_DL kernel matrices...


Computing K_D kernel matrices...


Train kernel matrix shape: (1500, 1500)
Test kernel matrix shape:  (500, 1500)


**Code Explanation:** We pre-compute the full kernel matrices and pass them to SVM via `kernel='precomputed'`. This matches the paper's implementation strategy described in **Section 4**, where they note that caching base kernel values $k(x_i, x_j)$ avoids redundant computation during SVM training.

## Step 4: Train SVMs with Both Kernels and Compare

In [5]:
C = 1.0  # SVM regularisation parameter

# Balanced Pairwise SVM (proposed method — K_DL)
# Reference: Section 3 + Section 5 (experimental setup)
svm_kdl = SVC(kernel='precomputed', C=C, probability=True, random_state=42)
svm_kdl.fit(K_DL_train, y_tr)
pred_kdl = svm_kdl.predict(K_DL_test)
prob_kdl = svm_kdl.predict_proba(K_DL_test)[:, 1]
acc_kdl  = accuracy_score(y_te, pred_kdl)
auc_kdl  = roc_auc_score(y_te, prob_kdl)

# Baseline (non-symmetric) SVM with K_D
# Reference: Section 2, Equation (4) — the un-balanced version
svm_kd = SVC(kernel='precomputed', C=C, probability=True, random_state=42)
svm_kd.fit(K_D_train, y_tr)
pred_kd = svm_kd.predict(K_D_test)
prob_kd = svm_kd.predict_proba(K_D_test)[:, 1]
acc_kd  = accuracy_score(y_te, pred_kd)
auc_kd  = roc_auc_score(y_te, prob_kd)

print("========== Results ==========")
print(f"K_DL (Balanced — Paper's Method):  Acc={acc_kdl:.4f}, AUC-ROC={auc_kdl:.4f}")
print(f"K_D  (Plain — Baseline):           Acc={acc_kd:.4f},  AUC-ROC={auc_kd:.4f}")
print(f"Improvement (Acc): +{acc_kdl - acc_kd:.4f}")
print(f"Improvement (AUC): +{auc_kdl - auc_kd:.4f}")

========== Results ==========
K_DL (Balanced — Paper's Method):  Acc=0.5500, AUC-ROC=0.4901
K_D  (Plain — Baseline):           Acc=0.5500,  AUC-ROC=0.5511
Improvement (Acc): +0.0000
Improvement (AUC): +-0.0610


**Code Explanation:** Both SVMs use `kernel='precomputed'` so the kernel matrix is passed directly (matching the LIBSVM setup described in **Section 4**). The balanced kernel $K_{DL}$ is the proposed method; $K_D$ is the non-symmetric baseline. We compare accuracy and AUC-ROC to verify that the balanced kernel outperforms the plain one, which is the key claim in **Section 5 (Table 1)** of the paper.

In [6]:
# Classification report
print("K_DL Classification Report:")
print(classification_report(y_te, pred_kdl, target_names=['Different', 'Same']))

print("K_D Classification Report:")
print(classification_report(y_te, pred_kd, target_names=['Different', 'Same']))

K_DL Classification Report:
              precision    recall  f1-score   support

   Different       0.55      0.54      0.55       250
        Same       0.55      0.56      0.55       250

    accuracy                           0.55       500
   macro avg       0.55      0.55      0.55       500
weighted avg       0.55      0.55      0.55       500

K_D Classification Report:
              precision    recall  f1-score   support

   Different       0.56      0.50      0.53       250
        Same       0.55      0.60      0.57       250

    accuracy                           0.55       500
   macro avg       0.55      0.55      0.55       500
weighted avg       0.55      0.55      0.55       500

